In [1]:
%cd /drive2/ryusejong/LFF
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
import json 
import time 
import re
import random
import types
import math
import numpy as np 
from tqdm.auto import tqdm
from util.utils import set_seed, read_data, save_result, get_answer_from_text, chat_huggingface, chat_huggingface_with_hidden_states, construct_conversation
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel


seed = 42
set_seed(seed)

/drive2/ryusejong/LFF


/drive2/ryusejong/miniconda3/envs/llm1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### incorrect case
question = "Bryan starts exercising at home during quarantine. To start, he decides to do 3 sets of 15 push-ups each. Near the end of the third set, he gets tired and does 5 fewer push-ups. How many push-ups did he do in total? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break it down step-by-step!\n\n1. Bryan starts with 3 sets of 15 push-ups each. So, he does 3 x 15 = 45 push-ups in the first two sets.\n2. In the third set, he does 5 fewer push-ups than usual. So, he does 15 - 5 = 10 push-ups in the third set.\n3. To find the total number of push-ups, we add the number of push-ups in the first two sets (45) to the number of push-ups in the third set (10).\n\n45 + 10 = 55\n\n## 55 ##\n\nSo, Bryan did a total of 55 push-ups."

### incorrect (miss condition) step index : 1
reasoning_steps = [l.strip() for l in reasoning.split("\n") if l.strip()]
print(f"# steps: {len(reasoning_steps)}")
prm_input_text = question + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps) + ' \n\n\n\n'

# steps: 7


In [3]:
### laod prm model
prm_path = "UW-Madison-Lee-Lab/Llama-PRM800K"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

candidate_tokens = [12, 10]
prm_tokenizer = AutoTokenizer.from_pretrained(prm_path)
prm_tokenizer.pad_token = prm_tokenizer.eos_token
prm_tokenizer.padding_side = 'left' 
prm_tokenizer.truncation_side = 'left'
    
prm = AutoModelForCausalLM.from_pretrained(
    prm_path,
    torch_dtype=torch.bfloat16,
    device_map=device
)
prm.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.60s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
    (rotary_

In [ ]:
### original score
with torch.no_grad():
    prm_input = torch.tensor([tokenizer.encode(prm_input_text)]).to(prm.device)
    prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
    #print(logits.shape)
    prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
    #print(scores.shape)
    step_scores = prm_scores[prm_input == 23535]
    step_probs  = step_scores.tolist()